# E2 v2.1 — Strong Calibration Benchmark
## Fresh Paired Replication for Calib-BERT v2

This revision is used because exact retraining of the historical canonical M0 run is not reproducible in the current Colab runtime, despite matching the archived data, split, GPU class, Torch/CUDA stack, and major library versions.

### Scientific rule
The archived APIN run remains **frozen historical evidence**.  
E2 v2.1 is a **new, predeclared paired replication**:

- retrain **M0 Vanilla BERT** and **M1 LS ε=0.10** from the same pretrained checkpoint,
- use the same archived split assignment,
- reset the same seed immediately before each paired training run,
- fit all post-hoc calibrators only on the untouched calibration subset (n=423),
- evaluate all predeclared conditions on the official test set only after the E2 lock is written,
- do **not** select a method because it looks best on the test set.

This avoids mixing one historical canonical model with one fresh runtime model.


In [ ]:
# ============================================================
# 0. USER CONFIGURATION
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path("/content/calibbert_v2_e2")   # change if needed
INPUT_DIR = PROJECT_ROOT / "inputs"
WORK_DIR = PROJECT_ROOT / "work"
OUT_DIR = PROJECT_ROOT / "outputs_e2"

for p in [INPUT_DIR, WORK_DIR, OUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Put the three files below in INPUT_DIR.
# Filenames may differ; the notebook auto-detects the CSVs by SHA-256.
CANONICAL_ZIP_HINT = INPUT_DIR / "calibbert Data Complete.zip"

EXPECTED_TRAIN_SHA256 = "4e791e60d3f04dd9f14abb1228b0d4f92bf628cc84097c1c173401e98d355f4e"
EXPECTED_TEST_SHA256  = "a3b6c9bf5d685f49d57e05d24266dddc52999843256b13e65c07267c543175fe"

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 3
WARMUP_STEPS = 0
GRAD_CLIP = 1.0
PRIMARY_SEED = 42

# Strictly canonical software versions from the archived environment:
CANONICAL_VERSIONS = {
    "python": "3.12.13",
    "torch": "2.11.0",
    "transformers": "5.13.1",
    "scikit_learn": "1.6.1",
    "pandas": "2.2.2",
    "numpy": "2.0.2",
}


## 1. Optional strict-environment setup

For exact M0 retraining, the closest possible recreation of the canonical environment is strongly preferred.  
The notebook itself later audits versions and will flag differences.

> If installing exact versions in Colab causes dependency conflicts, use a fresh runtime rather than modifying a runtime that already contains incompatible packages.


In [ ]:
# OPTIONAL — run only in a fresh compatible runtime if needed.
# %pip install -q "torch==2.11.0" "transformers==5.13.1" \
#     "scikit-learn==1.6.1" "pandas==2.2.2" "numpy==2.0.2" \
#     "scipy==1.16.3" "statsmodels==0.14.6" "matplotlib==3.10.8"


## 2. Imports, deterministic setup, and input auto-detection

In [ ]:
import os, sys, json, math, random, hashlib, zipfile, warnings, platform
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import sklearn
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    log_loss,
)

import scipy
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, softmax

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_all_seeds(PRIMARY_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk_size), b""):
            h.update(block)
    return h.hexdigest()

def find_file_by_sha256(root, expected_hash, suffix=".csv"):
    hits = []
    for p in Path(root).glob(f"*{suffix}"):
        try:
            if sha256_file(p) == expected_hash:
                hits.append(p)
        except Exception:
            pass
    if len(hits) != 1:
        raise RuntimeError(f"Expected exactly one {suffix} with hash {expected_hash}; found {hits}")
    return hits[0]

TRAIN_PATH = find_file_by_sha256(INPUT_DIR, EXPECTED_TRAIN_SHA256, ".csv")
TEST_PATH  = find_file_by_sha256(INPUT_DIR, EXPECTED_TEST_SHA256, ".csv")

# Find a ZIP containing the canonical manifest.
zip_candidates = list(INPUT_DIR.glob("*.zip"))
CANONICAL_ZIP = None
for zp in zip_candidates:
    try:
        with zipfile.ZipFile(zp) as z:
            if "MANIFEST.json" in z.namelist() and "development_split_assignment.csv" in z.namelist():
                CANONICAL_ZIP = zp
                break
    except Exception:
        pass

if CANONICAL_ZIP is None:
    raise FileNotFoundError("Canonical evidence ZIP with MANIFEST.json was not found.")

print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)
print("ZIP  :", CANONICAL_ZIP)


## 3. Extract and verify the frozen canonical evidence package

In [ ]:
CANONICAL_DIR = WORK_DIR / "canonical_evidence"
CANONICAL_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(CANONICAL_ZIP) as z:
    z.extractall(CANONICAL_DIR)

manifest = json.loads((CANONICAL_DIR / "MANIFEST.json").read_text())
print(json.dumps(manifest, indent=2))

assert manifest["input_hashes"]["dreaddit_train_sha256"] == EXPECTED_TRAIN_SHA256
assert manifest["input_hashes"]["dreaddit_test_sha256"] == EXPECTED_TEST_SHA256
assert sha256_file(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
assert sha256_file(TEST_PATH) == EXPECTED_TEST_SHA256

print("RAW INPUT HASHES MATCH THE CANONICAL APIN RUN.")


## 4. Environment audit

A version mismatch does not automatically invalidate probability-calibrator fitting, but **M0 retraining** should be treated as canonical only after its official-test metrics reproduce the archived M0 metrics.


In [ ]:
environment = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "transformers": __import__("transformers").__version__,
    "scikit_learn": sklearn.__version__,
    "scipy": scipy.__version__,
    "pandas": pd.__version__,
    "numpy": np.__version__,
}
print(json.dumps(environment, indent=2))
(OUT_DIR / "e2_environment.json").write_text(json.dumps(environment, indent=2))

print("\nCanonical environment:")
print(json.dumps(json.loads((CANONICAL_DIR / "environment.json").read_text()), indent=2))


## 5. Load raw data and **archived split assignment**

The archived assignment is authoritative. We do not regenerate `StratifiedGroupKFold` here.


In [ ]:
train_full = pd.read_csv(TRAIN_PATH)
test_full = pd.read_csv(TEST_PATH)
split_assignment = pd.read_csv(CANONICAL_DIR / "development_split_assignment.csv")

assert len(train_full) == 2838
assert len(test_full) == 715
assert len(split_assignment) == 2838

# Strong row-by-row identity checks against archived assignment.
for col in ["id", "post_id", "label"]:
    a = train_full[col].astype(str).reset_index(drop=True)
    b = split_assignment[col].astype(str).reset_index(drop=True)
    assert a.equals(b), f"Mismatch in canonical row identity for {col}"

# confidence is numeric; compare tightly.
assert np.allclose(
    train_full["confidence"].astype(float).values,
    split_assignment["confidence"].astype(float).values,
    rtol=0, atol=1e-12
)

print(split_assignment["split"].value_counts())
print(split_assignment.groupby("split")["label"].value_counts().sort_index())

def rows_for_split(name):
    idx = split_assignment.loc[split_assignment["split"] == name, "row_index"].to_numpy()
    return train_full.iloc[idx].reset_index(drop=True)

fit_df = rows_for_split("fit")
validation_df = rows_for_split("validation")
calibration_df = rows_for_split("calibration")

assert len(fit_df) == 1994
assert len(validation_df) == 421
assert len(calibration_df) == 423

train_final = pd.concat([fit_df, validation_df], ignore_index=True)
assert len(train_final) == 2415

print("fit/validation/calibration:", len(fit_df), len(validation_df), len(calibration_df))
print("AALS a_min from fit:", float(fit_df["confidence"].min()))


## 6. Exact canonical dataset/training helpers

In [ ]:
# Selected checkpoint supplies its own tokenizer files.
SELECTED_CKPT_DIR = CANONICAL_DIR / "selected_model_checkpoint"
tokenizer = AutoTokenizer.from_pretrained(SELECTED_CKPT_DIR)

class DreadditDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True).copy()
        enc = tokenizer(
            self.df["text"].astype(str).tolist(),
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(self.df["label"].astype(int).values, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }

def make_loader(dataset, batch_size, shuffle, seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        generator=g,
    )

train_final_ds = DreadditDataset(train_final)
calibration_ds = DreadditDataset(calibration_df)
test_ds = DreadditDataset(test_full)

@torch.no_grad()
def predict_logits(model, dataset, seed=PRIMARY_SEED):
    loader = make_loader(dataset, EVAL_BATCH_SIZE, False, seed)
    model.eval()
    logits_list, labels_list = [], []
    for batch in loader:
        outputs = model(
            input_ids=batch["input_ids"].to(DEVICE),
            attention_mask=batch["attention_mask"].to(DEVICE),
        )
        logits_list.append(outputs.logits.detach().cpu().numpy())
        labels_list.append(batch["labels"].numpy())
    return np.vstack(logits_list), np.concatenate(labels_list)

def train_vanilla_canonical(seed=PRIMARY_SEED):
    set_all_seeds(seed)
    train_loader = make_loader(train_final_ds, TRAIN_BATCH_SIZE, True, seed)

    # Exact optimizer structure used by the canonical notebook:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            loss = F.cross_entropy(outputs.logits, batch["labels"].to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.detach().cpu()))
        history.append({"epoch": epoch, "train_loss": float(np.mean(losses))})
        print(f"M0 epoch {epoch}/{EPOCHS}: loss={np.mean(losses):.6f}")
    return model, pd.DataFrame(history)


## 7. Fresh paired training: M0 Vanilla and M1 LS ε=0.10

Both models are trained **fresh in the same runtime**. The random state is reset immediately before each run. This is the correct paired comparison for E2 v2.1.


In [ ]:
# ============================================================
# 7. FRESH PAIRED TRAINING — M0 AND M1
# ============================================================
def soft_target_cross_entropy(logits, labels, epsilon):
    K = logits.shape[-1]
    eps = torch.full((labels.shape[0],), float(epsilon), device=labels.device)
    target = torch.full((labels.shape[0], K), 0.0, dtype=torch.float32, device=labels.device)
    target += (eps / (K - 1)).unsqueeze(1)
    target.scatter_(1, labels.unsqueeze(1), (1.0 - eps).unsqueeze(1))
    return -(target * F.log_softmax(logits, dim=1)).sum(dim=1).mean()

def train_paired_model(mode, epsilon, seed=PRIMARY_SEED):
    set_all_seeds(seed)
    train_loader = make_loader(train_final_ds, TRAIN_BATCH_SIZE, True, seed)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=WARMUP_STEPS,
        num_training_steps=total_steps,
    )

    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            outputs = model(
                input_ids=batch["input_ids"].to(DEVICE),
                attention_mask=batch["attention_mask"].to(DEVICE),
            )
            labels = batch["labels"].to(DEVICE)

            if mode == "none":
                loss = F.cross_entropy(outputs.logits, labels)
            elif mode == "ls":
                loss = soft_target_cross_entropy(outputs.logits, labels, epsilon)
            else:
                raise ValueError(mode)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
            scheduler.step()
            losses.append(float(loss.detach().cpu().item()))

        history.append({
            "epoch": epoch,
            "mode": mode,
            "epsilon": epsilon,
            "seed": seed,
            "train_loss": float(np.mean(losses)),
        })
        print(f"{mode} eps={epsilon} epoch {epoch}/{EPOCHS}: loss={np.mean(losses):.6f}")

    cal_logits, cal_y = predict_logits(model, calibration_ds, seed)
    test_logits, test_y = predict_logits(model, test_ds, seed)
    return model, pd.DataFrame(history), cal_logits, cal_y, test_logits, test_y

print("Training paired M0...")
m0_model, m0_history, m0_cal_logits, y_cal_m0, m0_test_logits_runtime, y_test_m0 = train_paired_model(
    mode="none", epsilon=0.0, seed=PRIMARY_SEED
)

print("\nTraining paired M1 LS 0.10...")
m1_model, m1_history, m1_cal_logits, y_cal, m1_test_logits_runtime, y_test = train_paired_model(
    mode="ls", epsilon=0.10, seed=PRIMARY_SEED
)

assert np.array_equal(y_cal_m0, y_cal)
assert np.array_equal(y_test_m0, y_test)

pd.concat([m0_history.assign(model="M0_Vanilla_BERT"),
           m1_history.assign(model="M1_LS_0.10")], ignore_index=True).to_csv(
    OUT_DIR / "e2_fresh_paired_training_history.csv", index=False
)

np.save(OUT_DIR / "M0_calibration_logits.npy", m0_cal_logits)
np.save(OUT_DIR / "M0_test_logits.npy", m0_test_logits_runtime)
np.save(OUT_DIR / "M1_LS010_calibration_logits.npy", m1_cal_logits)
np.save(OUT_DIR / "M1_LS010_test_logits.npy", m1_test_logits_runtime)

del m0_model, m1_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


## 8. Archived APIN reference audit — descriptive only

The archived canonical APIN results are preserved as historical reference.  
A mismatch here **does not stop E2 v2.1**, because both E2 comparison models were freshly retrained as a paired experiment in the same runtime.


In [ ]:
# ============================================================
# 8. HISTORICAL REFERENCE AUDIT — NO REPRODUCTION GATE
# ============================================================
archived_m0_logits = np.load(CANONICAL_DIR / "test_logits_M0_Vanilla_BERT.npy")
archived_m1_logits = np.load(CANONICAL_DIR / "test_logits_M1_LS_0.10.npy")

np.save(OUT_DIR / "archived_M0_test_logits_reference.npy", archived_m0_logits)
np.save(OUT_DIR / "archived_M1_test_logits_reference.npy", archived_m1_logits)

print("Archived APIN logits retained as reference only.")


## 9. Fresh paired baseline audit

This reports the new E2 paired M0 and M1 results and compares them descriptively with the archived APIN values. It does not tune or select any model.


In [ ]:
# ============================================================
# 9. METRICS + FRESH-PAIR AUDIT
# ============================================================
def probs_from_logits(logits):
    return softmax(logits, axis=1)

def ece_equal_width(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    edges = np.linspace(0, 1, n_bins+1)
    ece = 0.0
    for b in range(n_bins):
        lo, hi = edges[b], edges[b+1]
        mask = (conf >= lo) & ((conf <= hi) if b == n_bins-1 else (conf < hi))
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def ace_equal_frequency(y_true, probs, n_bins=10):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    correct = (pred == y_true).astype(float)
    order = np.argsort(conf)
    ace = 0.0
    for idx in np.array_split(order, n_bins):
        if len(idx):
            ace += (len(idx)/len(y_true)) * abs(correct[idx].mean() - conf[idx].mean())
    return float(ace)

def metric_bundle(y_true, probs):
    pred = probs.argmax(axis=1)
    conf = probs.max(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", pos_label=1, zero_division=0
    )
    cm = confusion_matrix(y_true, pred, labels=[0,1])
    return {
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "brier": float(np.mean((probs[:,1] - y_true)**2)),
        "nll": float(log_loss(y_true, probs, labels=[0,1])),
        "ece": ece_equal_width(y_true, probs, 10),
        "ace": ace_equal_frequency(y_true, probs, 10),
        "hce_0.90": int(((pred != y_true) & (conf >= 0.90)).sum()),
        "coverage_0.90": float((conf >= 0.90).mean()),
        "selective_error_0.90": float((pred[conf>=0.90] != y_true[conf>=0.90]).mean()) if (conf>=0.90).any() else np.nan,
        "tn": int(cm[0,0]), "fp": int(cm[0,1]), "fn": int(cm[1,0]), "tp": int(cm[1,1]),
    }

fresh_m0 = metric_bundle(y_test, probs_from_logits(m0_test_logits_runtime))
fresh_m1 = metric_bundle(y_test, probs_from_logits(m1_test_logits_runtime))
arch_m0 = metric_bundle(y_test, probs_from_logits(archived_m0_logits))
arch_m1 = metric_bundle(y_test, probs_from_logits(archived_m1_logits))

baseline_audit = pd.DataFrame([
    {"run":"E2_fresh", "model":"M0_Vanilla_BERT", **fresh_m0},
    {"run":"E2_fresh", "model":"M1_LS_0.10", **fresh_m1},
    {"run":"APIN_archived", "model":"M0_Vanilla_BERT", **arch_m0},
    {"run":"APIN_archived", "model":"M1_LS_0.10", **arch_m1},
])
baseline_audit.to_csv(OUT_DIR / "e2_fresh_vs_archived_baseline_audit.csv", index=False)
display(baseline_audit)


## 10. Predeclared post-hoc calibrators

### Temperature scaling
Fits one positive scalar `T` on the 2-class logits by minimizing calibration NLL.

### Platt scaling
Fits an unregularized logistic map on the raw logit margin `z1-z0`.

### Beta calibration
Fits the monotonic beta-calibration form:

`sigmoid(a log p - b log(1-p) + c)`, with `a,b >= 0`.

### Isotonic regression
Fits a nonparametric monotonic probability map on `p(stress)`.

All fitting below uses **only** `calibration_df` (n=423).


In [ ]:
EPS = 1e-12

def binary_nll(y, p):
    p = np.clip(np.asarray(p, float), EPS, 1-EPS)
    y = np.asarray(y, int)
    return float(-np.mean(y*np.log(p) + (1-y)*np.log(1-p)))

class TemperatureCalibrator:
    def fit(self, logits, y):
        def objective(log_T):
            T = np.exp(log_T)
            p = softmax(logits / T, axis=1)[:,1]
            return binary_nll(y, p)
        res = minimize_scalar(objective, bounds=(-4,4), method="bounded")
        self.T = float(np.exp(res.x))
        return self
    def predict_proba(self, logits):
        return softmax(logits / self.T, axis=1)
    def params(self):
        return {"T": self.T}

class PlattCalibrator:
    def fit(self, logits, y):
        s = logits[:,1] - logits[:,0]
        def objective(theta):
            a, b = theta
            return binary_nll(y, expit(a*s + b))
        res = minimize(objective, x0=np.array([1.0,0.0]), method="BFGS")
        self.a, self.b = map(float, res.x)
        self.success = bool(res.success)
        return self
    def predict_proba(self, logits):
        s = logits[:,1] - logits[:,0]
        q = expit(self.a*s + self.b)
        return np.column_stack([1-q, q])
    def params(self):
        return {"a":self.a, "b":self.b, "success":self.success}

class BetaCalibrator:
    def fit(self, logits, y):
        p = np.clip(softmax(logits, axis=1)[:,1], 1e-8, 1-1e-8)
        x1 = np.log(p)
        x2 = -np.log(1-p)
        def objective(theta):
            a,b,c = theta
            q = expit(a*x1 + b*x2 + c)
            return binary_nll(y, q)
        res = minimize(
            objective,
            x0=np.array([1.0,1.0,0.0]),
            method="L-BFGS-B",
            bounds=[(0,None),(0,None),(None,None)]
        )
        self.a, self.b, self.c = map(float, res.x)
        self.success = bool(res.success)
        return self
    def predict_proba(self, logits):
        p = np.clip(softmax(logits, axis=1)[:,1], 1e-8, 1-1e-8)
        q = expit(self.a*np.log(p) + self.b*(-np.log(1-p)) + self.c)
        return np.column_stack([1-q, q])
    def params(self):
        return {"a":self.a, "b":self.b, "c":self.c, "success":self.success}

class IsotonicCalibrator:
    def fit(self, logits, y):
        p = softmax(logits, axis=1)[:,1]
        self.iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        self.iso.fit(p, y)
        return self
    def predict_proba(self, logits):
        p = softmax(logits, axis=1)[:,1]
        q = np.clip(self.iso.predict(p), 0, 1)
        return np.column_stack([1-q, q])
    def params(self):
        return {
            "x_thresholds": self.iso.X_thresholds_.tolist(),
            "y_thresholds": self.iso.y_thresholds_.tolist(),
        }

CALIBRATOR_FACTORIES = {
    "temperature": TemperatureCalibrator,
    "platt": PlattCalibrator,
    "beta": BetaCalibrator,
    "isotonic": IsotonicCalibrator,
}


## 11. Fit all calibrators on the untouched calibration subset and write the E2 lock

No official-test metric is used for method fitting or method selection.


In [ ]:
MODEL_LOGITS = {
    "M0_Vanilla_BERT": {
        "calibration": m0_cal_logits,
        "test": m0_test_logits_runtime,
    },
    "M1_LS_0.10": {
        "calibration": m1_cal_logits,
        "test": m1_test_logits_runtime,
    },
}

fitted = {}
fit_rows = []
parameter_record = {}

for model_name, parts in MODEL_LOGITS.items():
    fitted[model_name] = {}
    parameter_record[model_name] = {}

    base_cal_probs = probs_from_logits(parts["calibration"])
    fit_rows.append({
        "model": model_name,
        "calibrator": "none",
        **metric_bundle(y_cal, base_cal_probs),
    })

    for cal_name, factory in CALIBRATOR_FACTORIES.items():
        cal = factory().fit(parts["calibration"], y_cal)
        fitted[model_name][cal_name] = cal
        parameter_record[model_name][cal_name] = cal.params()

        cal_probs = cal.predict_proba(parts["calibration"])
        fit_rows.append({
            "model": model_name,
            "calibrator": cal_name,
            **metric_bundle(y_cal, cal_probs),
        })

fit_metrics = pd.DataFrame(fit_rows)
fit_metrics.to_csv(OUT_DIR / "e2_calibration_fit_metrics.csv", index=False)
(OUT_DIR / "e2_calibrator_parameters.json").write_text(
    json.dumps(parameter_record, indent=2)
)

lock = {
    "run_name": "CalibBERT_v2_E2_v2_1_fresh_paired",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "canonical_input_hashes": manifest["input_hashes"],
    "development_split_source": "archived development_split_assignment.csv",
    "fit_partition": "calibration only (n=423)",
    "models": ["fresh_M0_Vanilla_BERT", "fresh_M1_LS_0.10"],
    "calibrators": ["none", "temperature", "platt", "beta", "isotonic"],
    "primary_probability_endpoints": ["brier", "nll"],
    "secondary_endpoints": ["ece", "ace", "hce_0.90", "coverage_0.90", "selective_error_0.90"],
    "automatic_test_based_method_selection": False,
    "official_test_metrics_used_for_calibrator_fit": False,
}
(OUT_DIR / "E2_CALIBRATION_LOCK.json").write_text(json.dumps(lock, indent=2))
print(json.dumps(lock, indent=2))
display(fit_metrics[["model","calibrator","brier","nll","ece","ace","f1"]])


## 12. Official-test benchmark after the E2 lock

This cell evaluates all **predeclared** calibration conditions.  
Results are descriptive benchmark evidence; the notebook intentionally does **not** declare a “winner” based on the test set.


In [ ]:
test_rows = []
prediction_exports = []

for model_name, parts in MODEL_LOGITS.items():
    base_probs = probs_from_logits(parts["test"])
    base_pred = base_probs.argmax(axis=1)

    conditions = [("none", base_probs)]
    for cal_name, cal in fitted[model_name].items():
        conditions.append((cal_name, cal.predict_proba(parts["test"])))

    for cal_name, probs in conditions:
        met = metric_bundle(y_test, probs)
        pred = probs.argmax(axis=1)
        test_rows.append({
            "model": model_name,
            "calibrator": cal_name,
            "decision_changes_vs_uncalibrated": int((pred != base_pred).sum()),
            **met,
        })

        prediction_exports.append(pd.DataFrame({
            "row_index": np.arange(len(test_full)),
            "id": test_full["id"].values,
            "post_id": test_full["post_id"].values,
            "true_label": y_test,
            "model": model_name,
            "calibrator": cal_name,
            "p_not_stress": probs[:,0],
            "p_stress": probs[:,1],
            "predicted_label": pred,
            "confidence": probs.max(axis=1),
        }))

test_results = pd.DataFrame(test_rows)
test_results.to_csv(OUT_DIR / "e2_official_test_calibration_benchmark.csv", index=False)

pred_long = pd.concat(prediction_exports, ignore_index=True)
pred_long.to_csv(OUT_DIR / "e2_prediction_level_long.csv", index=False)

display(
    test_results[
        ["model","calibrator","brier","nll","ece","ace","f1",
         "hce_0.90","coverage_0.90","selective_error_0.90",
         "decision_changes_vs_uncalibrated"]
    ].sort_values(["model","calibrator"])
)


## 13. Relative deltas against each model's uncalibrated condition

Negative ΔBrier / ΔNLL / ΔECE / ΔACE means improvement.


In [ ]:
delta_rows = []
for model_name in test_results["model"].unique():
    base = test_results[
        (test_results["model"] == model_name) &
        (test_results["calibrator"] == "none")
    ].iloc[0]

    for _, row in test_results[test_results["model"] == model_name].iterrows():
        delta_rows.append({
            "model": model_name,
            "calibrator": row["calibrator"],
            "delta_brier": row["brier"] - base["brier"],
            "delta_nll": row["nll"] - base["nll"],
            "delta_ece": row["ece"] - base["ece"],
            "delta_ace": row["ace"] - base["ace"],
            "delta_f1": row["f1"] - base["f1"],
            "delta_hce_0.90": row["hce_0.90"] - base["hce_0.90"],
            "delta_coverage_0.90": row["coverage_0.90"] - base["coverage_0.90"],
        })

deltas = pd.DataFrame(delta_rows)
deltas.to_csv(OUT_DIR / "e2_deltas_vs_uncalibrated.csv", index=False)
display(deltas)


## 14. Publication-ready descriptive table

Primary evidence is probability quality:
- Brier
- NLL

ECE/ACE remain secondary because bin-based calibration estimates are estimator-sensitive.


In [ ]:
publication_table = test_results[
    ["model","calibrator","accuracy","precision","recall","f1",
     "brier","nll","ece","ace","hce_0.90","coverage_0.90",
     "selective_error_0.90","decision_changes_vs_uncalibrated"]
].copy()

publication_table.to_csv(OUT_DIR / "Table_E2_Strong_Calibration_Benchmark.csv", index=False)
display(publication_table)


## 15. E2 v2.1 completion gate

E2 v2.1 is complete only if:

- [ ] raw Dreaddit train/test hashes match the archived APIN manifest
- [ ] archived split assignment is used unchanged
- [ ] M0 and M1 LS 0.10 are both retrained fresh in the same runtime with paired seed reset
- [ ] all post-hoc calibrators use only the 423-example calibration subset
- [ ] `E2_CALIBRATION_LOCK.json` exists before official-test calibration benchmarking
- [ ] no method is selected because it looks best on the official test
- [ ] prediction-level outputs are saved for E5/E7
- [ ] archived APIN metrics are retained only as historical reference

After completion, freeze `outputs_e2/`. The next low-cost step is **E5 Risk–Coverage / AURC**, using the saved prediction-level probabilities without additional Transformer training.
